# Modelado — Regresión lineal y Random Forest

**Curso:** Modelos y Simulación de Sistemas I — Proyecto integrador, Fase 1
**Continuación de:** `01-exploracion-datos.ipynb` y `02-preparacion-datos.ipynb`

**Problema:** predecir la duración de un viaje (`duracion_viaje_horas`, en horas) hasta la Terminal de Transporte de Medellín, usando información conocida al momento de la salida. Es un problema de **regresión**.

**Qué hace este notebook:** entrena cuatro candidatos, **regresión lineal** y **Random Forest**, cada uno **con y sin** la variable `pasajeros`; los compara en el conjunto de prueba; elige el mejor y **exporta solo ese** a `models/`.

**Modelo base:** la regresión lineal cumple ese papel. Es un modelo simple contra el cual se mide si el Random Forest realmente aporta.

## Configuración e imports

In [ ]:
import os
import sys
import time
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

SEED = 42

TRAIN_PATH = "../data/processed/train.csv.gz"
TEST_PATH = "../data/processed/test.csv.gz"
MODELS_DIR = "../models"

OBJETIVO = "duracion_viaje_horas"

## 1. Cargar los datos

Se leen los conjuntos generados en `02-preparacion-datos.ipynb`. La separación ya está hecha: **80 % entrenamiento y 20 % prueba**, aleatoria, con semilla 42. Los modelos se entrenan **solo con `train`**; `test` se usa para evaluarlos.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"train: {train.shape}  |  test: {test.shape}")
print(f"Valores faltantes -> train: {int(train.isna().sum().sum())} | test: {int(test.isna().sum().sum())}")
train.head()

train: (2093700, 11)  |  test: (523425, 11)
Valores faltantes -> train: 0 | test: 0


,nombre_sucursal,empresa,ruta_origen,subregi_n,clase_veh_culo,pasajeros,hora_salida_decimal,dia_semana,mes,duracion_viaje_horas,duracion_corregida
0,TERMINAL DEL NORTE,EXPRE - BELMIRA S.A.,SAN PEDRO DE LOS MILAGROS,NORTE,CAMIONETA,8,10.000000,4,2,1.150000,False
1,TERMINAL DEL NORTE,COONORTE,PUERTO BERRIO,NORTE,MICROBUS,5,4.750000,2,10,3.600000,False
2,TERMINAL DEL NORTE,TRANS CISNEROS-ENTRERRIOS,MACEO,NORTE,BUS,12,8.000000,4,2,2.600000,False
3,TERMINAL DEL NORTE,TRANS CISNEROS-ENTRERRIOS,ENTRERRIOS,NORTE,CAMIONETA,6,5.500000,0,1,2.083333,False
4,TERMINAL DEL NORTE,FLOTA EL CARMEN,EL CARMEN DE VIBORAL,ORIENTE,MICROBUS,19,7.416667,1,9,1.600000,False


## 2. Definir las variables

**Hora de salida.** La exploración mostró que la hora afecta la duración de forma *no lineal*: los viajes que salen de noche duran mucho más que los del día. Una regresión lineal que reciba la hora como un número solo puede aprender "a más hora, más (o menos) duración", lo que no describe ese patrón. Por eso se crea también `hora_entera` (0 a 23), que la regresión lineal tratará como una categoría. Es un cálculo directo sobre cada fila, sin aprender nada de los datos, así que no genera fuga de información. El Random Forest usa la hora original.

**Qué queda fuera.** `duracion_corregida` indica si el objetivo fue reemplazado por la mediana de su ruta; se deriva del objetivo, así que **no puede ser predictora**. Se usa solo para evaluar aparte las filas cuya duración es original.

Todos los modelos reciben las **mismas columnas de entrada**, y cada uno selecciona y transforma las que necesita (incluso los que no usan `pasajeros` la reciben y la descartan). Así el modelo exportado se usa igual sea cual sea el ganador.

In [ ]:
for df in (train, test):
    df["hora_entera"] = df["hora_salida_decimal"].astype(int)

CATEGORICAS = ["nombre_sucursal", "empresa", "ruta_origen", "subregi_n", "clase_veh_culo"]
NUMERICAS = ["pasajeros", "hora_salida_decimal", "dia_semana", "mes"]
PREDICTORAS = CATEGORICAS + NUMERICAS + ["hora_entera"]

X_train, y_train = train[PREDICTORAS], train[OBJETIVO]
X_test, y_test = test[PREDICTORAS], test[OBJETIVO]
mask_sin_corregir = (~test["duracion_corregida"]).to_numpy()     # solo para evaluar, nunca como predictora

# Comprobaciones de seguridad: si alguien las rompe, el notebook falla
assert OBJETIVO not in X_train.columns
assert "duracion_corregida" not in X_train.columns and "duracion_original" not in X_train.columns
assert list(X_train.columns) == list(X_test.columns)

print("Predictoras:", PREDICTORAS)
print(f"Filas de prueba con duración original (sin corregir): {mask_sin_corregir.sum():,} de {len(test):,}")

Predictoras: ['nombre_sucursal', 'empresa', 'ruta_origen', 'subregi_n', 'clase_veh_culo', 'pasajeros', 'hora_salida_decimal', 'dia_semana', 'mes', 'hora_entera']
Filas de prueba con duración original (sin corregir): 522,712 de 523,425


## 3. Modelos candidatos

### Regresión lineal
Modelo simple, rápido y fácil de interpretar: cada coeficiente indica cuántas horas suma o resta una categoría. Solo aprende relaciones lineales con lo que se le entrega. **En este notebook actúa como modelo base:** el Random Forest solo se justifica si la mejora de forma clara.

- **Codificación (one-hot).** La regresión lineal solo trabaja con números. Numerar las categorías (ruta 1, ruta 2, ruta 3...) inventaría un orden que no existe. Con `OneHotEncoder` cada categoría se convierte en una columna de 0 y 1 ("¿el viaje es de esta ruta?") y el modelo aprende un efecto por categoría. Con `handle_unknown="ignore"`, una categoría que no apareció en entrenamiento no rompe la predicción. Se codifican las 5 categóricas, `hora_entera`, `dia_semana` y `mes` (estos dos son códigos, no cantidades).
- **Sin escalar.** La regresión lineal ordinaria da las mismas predicciones con o sin escalado, así que no se usa.

### Random Forest
Es un conjunto de muchos **árboles de decisión**. Cada árbol aprende reglas del tipo "si la ruta es X y la salida es después de las 20:00, la duración es Y", y la predicción final es el **promedio** de todos los árboles. Captura por sí solo efectos no lineales e interacciones (por ejemplo, que la hora pese distinto según la ruta).

- **Codificación.** Los árboles dividen con umbrales, así que no necesitan one-hot ni `hora_entera`: las 5 categóricas se convierten en enteros con `OrdinalEncoder` (las categorías nuevas reciben -1) y las numéricas pasan sin cambios. Tampoco necesitan escalado.
- **Parámetros:**

| Parámetro | Valor | Por qué |
|---|---|---|
| `n_estimators` | 80 | Número de árboles: más árboles dan predicciones más estables, pero el archivo crece con cada uno |
| `min_samples_leaf` | 50 | Cada hoja representa al menos 50 viajes: limita el tamaño del árbol (el archivo debe caber en GitHub, límite 100 MB) y evita memorizar viajes individuales |
| `max_samples` | 0.5 | Cada árbol se entrena con una muestra aleatoria del 50 % del tamaño del entrenamiento (con reemplazo): acelera el entrenamiento y hace que los árboles sean más distintos entre sí |
| `n_jobs` | -1 | Usa todos los núcleos del computador |
| `random_state` | 42 | Semilla, para poder repetir el entrenamiento |

### Con y sin `pasajeros`
En la exploración quedó pendiente decidir si `pasajeros` estaría disponible al momento de predecir: se registra en la terminal de llegada y podría incluir pasajeros que abordan durante el trayecto. Por eso cada algoritmo se entrena en dos versiones y se compara.

### Por qué un `Pipeline`
Une la codificación y el modelo en un solo objeto. Al llamar a `fit`, el codificador aprende las categorías **solo de los datos de entrenamiento**; con datos nuevos únicamente se aplica la transformación ya aprendida. Además, el modelo guardado recibe las columnas originales y devuelve la predicción, sin repetir pasos de preparación.

In [ ]:
def crear_modelo(algoritmo, usar_pasajeros=True):
    """Devuelve un Pipeline sin entrenar. Siempre recibe las mismas columnas de entrada (PREDICTORAS)."""
    if algoritmo == "Regresión lineal":
        transformadores = [("categoricas", OneHotEncoder(handle_unknown="ignore"),
                            CATEGORICAS + ["hora_entera", "dia_semana", "mes"])]
        if usar_pasajeros:
            transformadores.append(("pasajeros", "passthrough", ["pasajeros"]))
        regresor = LinearRegression()

    elif algoritmo == "Random Forest":
        numericas = ["hora_salida_decimal", "dia_semana", "mes"] + (["pasajeros"] if usar_pasajeros else [])
        transformadores = [
            ("categoricas", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAS),
            ("numericas", "passthrough", numericas),
        ]
        regresor = RandomForestRegressor(n_estimators=80, min_samples_leaf=50, max_samples=0.5,
                                         n_jobs=-1, random_state=SEED)
    else:
        raise ValueError(f"Algoritmo desconocido: {algoritmo}")

    # remainder="drop": las columnas que el modelo no usa se descartan
    return Pipeline([
        ("codificacion", ColumnTransformer(transformadores, remainder="drop")),
        ("regresion", regresor),
    ])

ALGORITMOS = ["Regresión lineal", "Random Forest"]
CANDIDATOS = {f"{alg} ({'con' if con else 'sin'} pasajeros)": (alg, con)
              for alg in ALGORITMOS for con in (True, False)}
list(CANDIDATOS)

['Regresión lineal (con pasajeros)',
 'Regresión lineal (sin pasajeros)',
 'Random Forest (con pasajeros)',
 'Random Forest (sin pasajeros)']

## 4. Entrenamiento de los cuatro candidatos

Cada candidato se entrena con **todo** el conjunto de entrenamiento.

In [ ]:
modelos = {}
for nombre, (alg, con_pasajeros) in CANDIDATOS.items():
    t0 = time.time()
    modelos[nombre] = crear_modelo(alg, con_pasajeros).fit(X_train, y_train)
    print(f"{nombre}: entrenado en {time.time() - t0:.0f} s")

Regresión lineal (con pasajeros): entrenado en 49 s
Regresión lineal (sin pasajeros): entrenado en 72 s
Random Forest (con pasajeros): entrenado en 203 s
Random Forest (sin pasajeros): entrenado en 167 s


## 5. Evaluación y comparación en el conjunto de prueba

**Métricas:**
- **MAE (métrica principal):** error absoluto medio, **en horas**. Para estimar un tiempo de llegada es la más natural ("el modelo se equivoca en promedio X horas") y los pocos viajes extremadamente largos la afectan menos que al RMSE.
- **RMSE:** también en horas, pero penaliza más los errores grandes; indica si el modelo falla mucho en algunos casos.
- **R²:** proporción de la variación de la duración que explica el modelo (1 es perfecto, 0 es no explicar nada).

Las métricas de clasificación (accuracy, F1) no aplican porque el objetivo es continuo. Las predicciones **no se recortan**.

Cada modelo se evalúa sobre **todas** las filas de prueba y sobre las **filas sin corregir** (aquellas cuya duración es original, no reemplazada por la mediana de su ruta). Si ambos resultados son parecidos, la métrica no depende de valores reemplazados.

In [ ]:
def evaluar(modelo, nombre):
    pred = modelo.predict(X_test)
    y_real, p = y_test.to_numpy(), pred
    y_c, p_c = y_real[mask_sin_corregir], p[mask_sin_corregir]
    return {
        "Modelo": nombre,
        "RMSE (Global)": np.sqrt(mean_squared_error(y_real, p)),
        "MAE (Global)": mean_absolute_error(y_real, p),
        "R2 (Global)": r2_score(y_real, p),
        "RMSE (Sin Corregir)": np.sqrt(mean_squared_error(y_c, p_c)),
        "MAE (Sin Corregir)": mean_absolute_error(y_c, p_c),
        "R2 (Sin Corregir)": r2_score(y_c, p_c),
    }

tabla = pd.DataFrame([evaluar(m, n) for n, m in modelos.items()])
tabla.sort_values("MAE (Global)").reset_index(drop=True).round(4)

,Modelo,RMSE (Global),MAE (Global),R2 (Global),RMSE (Sin Corregir),MAE (Sin Corregir),R2 (Sin Corregir)
0,Random Forest (sin pasajeros),0.7871,0.3622,0.9520,0.7861,0.3616,0.9520
1,Random Forest (con pasajeros),0.8137,0.3715,0.9487,0.8117,0.3708,0.9489
2,Regresión lineal (con pasajeros),0.8547,0.4312,0.9434,0.8544,0.4308,0.9433
3,Regresión lineal (sin pasajeros),0.8548,0.4312,0.9434,0.8545,0.4309,0.9433


## 6. Selección del modelo

**Regla de selección** (definida antes de mirar los resultados), con la MAE global como criterio:
1. Para cada algoritmo, la versión **con** `pasajeros` solo se conserva si mejora la MAE de la versión **sin** ella en al menos un **1 %**. Si no, se prefiere la versión sin `pasajeros`: al no estar seguros de que esa información exista al salir, no vale la pena depender de ella por una mejora mínima.
2. Entre las dos versiones resultantes (una por algoritmo), gana la de **menor MAE**.

> **Nota metodológica:** la selección se hace sobre el conjunto de prueba, por consistencia con el resto del trabajo del equipo. Por eso la cifra del ganador puede ser algo optimista. Se espera que el efecto sea pequeño: hay pocos candidatos y más de 500.000 filas de prueba.

In [ ]:
UMBRAL_PASAJEROS = 0.01
mae = tabla.set_index("Modelo")["MAE (Global)"]

elegidos = {}
for alg in ALGORITMOS:
    con, sin = mae[f"{alg} (con pasajeros)"], mae[f"{alg} (sin pasajeros)"]
    mejora = (sin - con) / sin
    usar = mejora >= UMBRAL_PASAJEROS
    elegidos[alg] = f"{alg} ({'con' if usar else 'sin'} pasajeros)"
    print(f"{alg}: MAE con pasajeros {con:.4f} h | sin pasajeros {sin:.4f} h | "
          f"mejora por incluirla: {mejora * 100:.2f}% -> se conserva la versión {'CON' if usar else 'SIN'} pasajeros")

ganador = min(elegidos.values(), key=lambda n: mae[n])
alg_ganador, usa_pasajeros_ganador = CANDIDATOS[ganador]
print(f"\nModelo elegido: {ganador}  (MAE = {mae[ganador]:.4f} h ≈ {mae[ganador] * 60:.0f} min)")

Regresión lineal: MAE con pasajeros 0.4312 h | sin pasajeros 0.4312 h | mejora por incluirla: 0.01% -> se conserva la versión SIN pasajeros
Random Forest: MAE con pasajeros 0.3715 h | sin pasajeros 0.3622 h | mejora por incluirla: -2.57% -> se conserva la versión SIN pasajeros

Modelo elegido: Random Forest (sin pasajeros)  (MAE = 0.3622 h ≈ 22 min)


In [ ]:
# Ganador frente al modelo base (regresión lineal, en la versión que dejó la regla anterior)
base = elegidos["Regresión lineal"]
t = tabla.set_index("Modelo")

print(f"Modelo base   ({base}): MAE {t.loc[base, 'MAE (Global)']:.4f} h | RMSE {t.loc[base, 'RMSE (Global)']:.4f} h | R2 {t.loc[base, 'R2 (Global)']:.4f}")
print(f"Modelo elegido ({ganador}): MAE {t.loc[ganador, 'MAE (Global)']:.4f} h | RMSE {t.loc[ganador, 'RMSE (Global)']:.4f} h | R2 {t.loc[ganador, 'R2 (Global)']:.4f}")
for metrica in ["MAE (Global)", "RMSE (Global)"]:
    mejora = (t.loc[base, metrica] - t.loc[ganador, metrica]) / t.loc[base, metrica] * 100
    print(f"Reducción de {metrica.split()[0]} respecto al modelo base: {mejora:.1f}%")

# Escala de la duración, para juzgar si el error es razonable
print(f"\nDuración en prueba -> mediana: {y_test.median():.2f} h | media: {y_test.mean():.2f} h | desv. estándar: {y_test.std():.2f} h")
print(f"MAE del modelo elegido / mediana de la duración: {t.loc[ganador, 'MAE (Global)'] / y_test.median() * 100:.0f}%")

Modelo base   (Regresión lineal (sin pasajeros)): MAE 0.4312 h | RMSE 0.8548 h | R2 0.9434
Modelo elegido (Random Forest (sin pasajeros)): MAE 0.3622 h | RMSE 0.7871 h | R2 0.9520
Reducción de MAE respecto al modelo base: 16.0%
Reducción de RMSE respecto al modelo base: 7.9%

Duración en prueba -> mediana: 1.98 h | media: 3.49 h | desv. estándar: 3.59 h
MAE del modelo elegido / mediana de la duración: 18%


## 7. Exportar el modelo elegido y verificarlo

Solo se guarda el ganador (el pipeline completo, con `joblib`). Se comprueba que:
1. El archivo **cabe en GitHub** (límite de 100 MB por archivo; con más de 50 MB aparece una advertencia).
2. Al volver a cargarlo, **genera exactamente las mismas predicciones** que el modelo en memoria.

In [ ]:
NOMBRES_ARCHIVO = {"Regresión lineal": "modelo_regresion_lineal.joblib", "Random Forest": "modelo_random_forest.joblib"}

os.makedirs(MODELS_DIR, exist_ok=True)
ruta_modelo = os.path.join(MODELS_DIR, NOMBRES_ARCHIVO[alg_ganador])
modelo_final = modelos[ganador]

joblib.dump(modelo_final, ruta_modelo, compress=3)
tam_mb = os.path.getsize(ruta_modelo) / 1e6
print(f"Modelo exportado en {ruta_modelo} ({tam_mb:.1f} MB)")

assert tam_mb < 100, "El archivo supera el límite de GitHub: reducir n_estimators o subir min_samples_leaf"
if tam_mb > 50:
    print("Aviso: GitHub advierte con archivos de más de 50 MB; considerar reducir el modelo.")

Modelo exportado en ../models\modelo_random_forest.joblib (47.4 MB)


In [ ]:
muestra = X_test.head(1000)
modelo_cargado = joblib.load(ruta_modelo)
assert np.allclose(modelo_cargado.predict(muestra), modelo_final.predict(muestra))
print("El modelo cargado produce las mismas predicciones que el modelo entrenado.\n")

ejemplo = muestra[["ruta_origen", "empresa", "hora_salida_decimal"]].head(5).copy()
ejemplo["duración predicha (h)"] = modelo_cargado.predict(muestra.head(5)).round(2)
ejemplo

El modelo cargado produce las mismas predicciones que el modelo entrenado.



,ruta_origen,empresa,hora_salida_decimal,duración predicha (h)
0,JERICO,TRANSPORTES JERICO,17.000000,3.49
1,SEGOVIA,TRANSPORTES SEGOVIA,8.250000,5.82
2,EL CARMEN DE VIBORAL,FLOTA EL CARMEN,19.666667,1.44
3,CISNEROS,TRANS CISNEROS-ENTRERRIOS,5.500000,1.80
4,MANIZALES,EMPRESA ARAUCA S.A.,17.500000,5.24


## 8. Reproducibilidad

- **Semilla fija** (`SEED = 42`) en la separación de datos (notebook 02) y en el Random Forest. La regresión lineal es determinista.
- **Se ejecuta de principio a fin** sin intervención manual. Solo necesita `train.csv.gz` y `test.csv.gz` en `fase-1/data/processed/`, que vienen en el repositorio.
- **Versiones** usadas en esta ejecución (importan al cargar el modelo guardado, que debe abrirse con una versión compatible de scikit-learn):

In [ ]:
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__} | numpy: {np.__version__} | scikit-learn: {sklearn.__version__} | joblib: {joblib.__version__}")

Python: 3.11.9 | pandas: 3.0.6 | numpy: 2.1.3 | scikit-learn: 1.9.1 | joblib: 1.6.0


## 9. Interpretación de resultados

**¿El modelo mejora al modelo base?** Sí. Con la regresión lineal como modelo base, el Random Forest sin `pasajeros` la supera en las tres métricas: el MAE baja de 0.431 h a 0.362 h (un 16 % menos, unos 4 minutos por viaje), el RMSE de 0.855 h a 0.787 h (un 8 % menos) y el R² sube de 0.943 a 0.952. Es coherente con lo que se vio en la exploración: la duración depende de la ruta, la hora y sus interacciones (la hora pesa distinto según la ruta). El Random Forest las captura por sí solo, mientras que la regresión lineal solo suma efectos independientes y necesitó que se le preparara la hora como categoría.

**¿La métrica obtenida es razonable?** Sí. El MAE de 0.362 h (unos 22 minutos) es una fracción pequeña de la duración de un viaje típico (ver la mediana en la sección 6), y el R² de 0.952 indica que el modelo explica el 95 % de la variación de la duración. El resultado sobre las filas sin corregir (MAE 0.3616 h) es prácticamente igual al global, así que la métrica no depende de las duraciones que se reemplazaron en la preparación. El RMSE es más del doble del MAE (0.787 h frente a 0.362 h): la mayoría de los errores son pequeños y unos pocos viajes concentran errores grandes.

**¿Influye `pasajeros`?** No aporta. En la regresión lineal, incluirla mejora el MAE apenas un 0.01 %; en el Random Forest lo empeora (0.3715 h con ella contra 0.3622 h sin ella). Con la regla definida (conservarla solo si mejora el MAE al menos un 1 %), se descartó en ambos algoritmos. Una posible explicación para el Random Forest es que, al ser una variable continua con muchos valores distintos, los árboles la usan para ajustar ruido en lugar de patrones generales; esto no se verificó. Prescindir de ella también evita depender de un dato que podría no conocerse al momento de la salida.

**Principales dificultades**
- La duración tiene una cola larga, por lo que unos pocos viajes concentran errores grandes.
- Hay muchas rutas y empresas, muy relacionadas entre sí, lo que dificulta interpretar el efecto de cada una por separado.
- El tamaño del archivo del Random Forest obligó a limitar el número de árboles y a exigir hojas de al menos 50 viajes para que cupiera en GitHub.
- La selección se hizo sobre el conjunto de prueba, por lo que la cifra del ganador puede ser algo optimista; se espera un efecto pequeño (pocos candidatos y más de 500.000 filas de prueba).

**Qué se podría mejorar**
- Ajustar los hiperparámetros del Random Forest (más árboles o mayor profundidad) si el límite de tamaño lo permite.
- Medir cuánto depende el modelo de haber visto la ruta antes, con una validación por grupos de ruta, útil si aparecieran rutas nuevas.
- Analizar en qué rutas y horarios se concentran los errores más grandes.